In [5]:
"""
Dual-vision Qwen3-VL integration (RGB + hyperspectral).

We inject two visual streams into Qwen’s language-model context:
  (1) Qwen’s native image-tower tokens from a pseudo-RGB image built from 3 HS bands
  (2) Our pretrained HS encoder tokens from the full 285-band cube

Key constraint:
  - HS tokens must have hidden size == Qwen text hidden size (e.g., 2048 for Qwen3-VL-2B)
  - We keep HS token count fixed (HS_NUM_TOKENS) for VRAM-controlled sweeps
"""

import sys
import numpy as np
import glob, random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

# Make local module importable
sys.path.append("/dataset/CSE290D_F25/transformer_head_work")
from hsi_models_v3_3c import HyperspectralVisionEncoder, load_band_stats

QWEN_NAME = "Qwen/Qwen3-VL-2B-Instruct"  #Qwen model to grab
HS_TOKEN = "<|hs_image_pad|>"            #Dedicated placeholder string for injecting HS tokens into Qwen
HS_NUM_TOKENS = 64                       #Number of HS tokens to inject, must match with HS encoder model

FILL = -9999.0
R_BAND, G_BAND, B_BAND = 36, 22, 10

DEVICE = "cuda:0"
DTYPE = torch.bfloat16

## HSI-Encoder Model weights to load:
ENCODER_CKPT_PATH = "/more_data/models/v3.3c/hs_autoencoder_best_v3.3c_16qx1024d_20260201_234607_3e-05-3e-07_val_loss_0.447045.pt"
BAND_STATS_PATH = "emit_band_stats.npz"

In [6]:
"""
Load Qwen3-VL (processor + model) and add a dedicated HS placeholder token.

We will later replace (<masked_scatter>) embeddings at HS_TOKEN positions with HS encoder features.
"""

processor = AutoProcessor.from_pretrained(QWEN_NAME)  #Grabs the tokenizer, image/video processor, and model specific tokens/templates
qwen = Qwen3VLForConditionalGeneration.from_pretrained(QWEN_NAME, dtype=DTYPE, device_map=None).to(DEVICE) #Load qwen model

processor.tokenizer.add_special_tokens({"additional_special_tokens": [HS_TOKEN]})  #Adds the dedicated HS placeholder token to the tokenizer
qwen.resize_token_embeddings(len(processor.tokenizer))  #Resize input embedding table to include the new token
HS_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids(HS_TOKEN)  # get the integer token ID corresponding to the HS placeholder token.

# Qwen RoPE indexing uses the image_processor merge size (typically 2)
ROPE_MERGE_SIZE = processor.image_processor.merge_size  #Grab updated merge size for token calculations later

In [7]:
"""
Load pretrained HS encoder weights saved from HS autoencoder training.

Checkpoint format (from custom_encoder.ipynb):
  ckpt = {
    "model_state_dict": ...  # full HyperspectralAutoencoder state_dict()
    "band_mean": ..., "band_std": ...,
    "patch": 16,
    "spatial_merge_size": 1,
    "hidden_size": hidden_size,
    ...
  }

We only need the encoder submodule weights ("hsi_encoder.*") for Qwen token injection.
"""

class TokenResampler(nn.Module):
    """
    Purpose: Learnable resampling module that converts an input sequence of tokens of length T 
    into a fixed length num_tokens using cross-attention.    
    """
    def __init__(self, hidden_size, num_tokens, num_heads=8):
        """
        Parameters:
        hidden_size: token embedding size (must match Qwen text hidden size later)
        num_tokens: desired output token count (you pass HS_NUM_TOKENS)
        num_heads: attention heads for the resampler MHA.  Ensure hidden_size % num_heads == 0.
        
        """
        super().__init__()
        self.q = nn.Parameter(torch.randn(1, num_tokens, hidden_size) * 0.02)  # Learnable "latent tokens" query bank, to make the resampler produce a fixed length output.  
                                                                               # Each query "extracts" one summary token from the input sequence.
                                                                               # The 0.02 value: Downscales random initialization values to aid in initial stability
        # Optional:  Add layer norm here if training is unstable
        self.attn = nn.MultiheadAttention(hidden_size, num_heads, batch_first=True)  # MHA over input tokens to act as resampling mechanism
                                                                                     # Lets each learned query token selectively aggregate information from the entire input HS token sequence
        self.ln = nn.LayerNorm(hidden_size)  # Normalize the output tokens

    def forward(self, x):
        """
        Take a variable-length sequence of input tokens x and return a fixed-length sequence of resampled tokens.
        """
        q = self.q.expand(x.size(0), -1, -1)
        y, _ = self.attn(q, x, x, need_weights=False)
        return self.ln(y)

class HSFixedTokens(nn.Module):
    """
    This module is an adapter: it wraps your variable-token-count HS backbone (HyperspectralVisionEncoder)
    and forces it to output exactly out_tokens HS tokens per sample,
    so you can reliably inject them into Qwen at the HS_TOKEN placeholder positions.
    
    """
    def __init__(self, hs_backbone, out_tokens, heads=8):
        """
        Purpose: Build the wrapper around a provided HS encoder.
        Inputs:
        hs_backbone: your HS encoder module. In this notebook it’s HyperspectralVisionEncoder(...).
        out_tokens: the desired fixed token count per sample (typically HS_NUM_TOKENS).
        heads: number of attention heads inside the resampler.
        """
        super().__init__()
        self.hs = hs_backbone
        self.hidden_size = hs_backbone.hidden_size
        self.resampler = TokenResampler(self.hidden_size, out_tokens, num_heads=heads)

    def forward(self, pixel_values, grid_thw=None, **kwargs):
        B = pixel_values.shape[0]
        out = self.hs(pixel_values, grid_thw=grid_thw, **kwargs)

        # New encoder returns (B, T, D). Old version returned (flat, deepstack).
        tokens = out[0] if isinstance(out, (tuple, list)) else out

        # Support both flattened (B*T, D) and unflattened (B, T, D)
        if tokens.dim() == 2:
            x = tokens.view(B, -1, self.hidden_size)
        else:
            x = tokens

        x = self.resampler(x)
        return x.reshape(-1, self.hidden_size), []  # (B*HS_NUM_TOKENS, D)

ckpt = torch.load(ENCODER_CKPT_PATH, map_location="cpu", weights_only=False)   #Load HSI Encoder weights

# Grab normalization stats:
band_mean = ckpt["band_mean"]
band_std  = ckpt["band_std"]

# HS encoder architecture must match training (depth/heads/mlp_ratio should match your training notebook defaults)

# Old Model Loading Code:
# if "depth" in ckpt and "num_heads" in ckpt and "mlp_ratio" in ckpt:
#     hs_backbone = HyperspectralVisionEncoder(
#         embed_dim=ckpt["hidden_size"],          # should match qwen.config.text_config.hidden_size
#         num_bands=285,   #num bands will always be 285 for EMIT data
#         patch_size=ckpt["patch"],
#         spatial_merge_size=ckpt["spatial_merge_size"],
#         depth=ckpt["depth"],
#         num_heads=ckpt["num_heads"],
#         mlp_ratio=ckpt["mlp_ratio"],
#     ).to(DEVICE, dtype=DTYPE)
# else:
#     hs_backbone = HyperspectralVisionEncoder(
#         embed_dim=ckpt["hidden_size"],          # should match qwen.config.text_config.hidden_size
#         num_bands=285,
#         patch_size=ckpt["patch"],
#         spatial_merge_size=ckpt["spatial_merge_size"],
#         depth=6,
#         num_heads=8,
#         mlp_ratio=4.0,
#     ).to(DEVICE, dtype=DTYPE)
# enc_sd = {k.replace("hsi_encoder.", ""): v for k, v in ckpt["model_state_dict"].items() if k.startswith("hsi_encoder.")}

# V3.3c model loading code:
hs_backbone = HyperspectralVisionEncoder(
    embed_dim=ckpt["hidden_size"],
    num_bands=285,
    num_spectral_queries=ckpt.get("num_spectral_queries", 4),
    patch_size_spatial=ckpt.get("patch_size_spatial", 16),
    patch_size_spectral=ckpt.get("patch_size_spectral", 15),
    depth=ckpt.get("depth", 6),
    num_heads=ckpt.get("num_heads", 8),
    mlp_ratio=ckpt.get("mlp_ratio", 4.0),
).to(DEVICE, dtype=DTYPE)

# Load only encoder weights from the full autoencoder state_dict
sd = ckpt["model_state_dict"]

# optional: strip prefixes if present
if len(sd) > 0 and next(iter(sd.keys())).startswith("module."):
    sd = {k[len("module."):]: v for k, v in sd.items()}
if len(sd) > 0 and next(iter(sd.keys())).startswith("_orig_mod."):
    sd = {k[len("_orig_mod."):]: v for k, v in sd.items()}

enc_sd = {k.replace("hsi_encoder.", ""): v for k, v in sd.items() if k.startswith("hsi_encoder.")}

# Load only encoder weights from the full autoencoder state_dict

hs_backbone.load_state_dict(enc_sd, strict=True)

# Wrap to fixed HS_NUM_TOKENS to standardize token count
# corrects mismatch between tokens generated by encoder and HS_NUM_TOKENS in a learnable (less destructive) way
hs_encoder = HSFixedTokens(hs_backbone, out_tokens=HS_NUM_TOKENS, heads=8).to(DEVICE, dtype=DTYPE)

# Precompute grid_thw for Qwen RoPE indexing of HS images (matches fixed HS_NUM_TOKENS)
hs_grid_thw = torch.tensor([[1, 1, HS_NUM_TOKENS * (ROPE_MERGE_SIZE**2)]], device=DEVICE, dtype=torch.long)

In [8]:
"""
This cell creates the core “dual vision injection” wrapper. Its job is:
- Build Qwen’s normal text embeddings inputs_embeds of shape (B, S, D)
- Replace (“inject”) some positions in those embeddings with:
  - RGB vision features from Qwen’s native image tower
  - HS vision features from your custom HS encoder (hs_encoder)
- Compute correct RoPE position_ids so Qwen’s language model can attend properly over mixed text+vision tokens
- Run the language model forward and optionally compute loss

Where:
B = batch size
S = total sequence length in tokens (text tokens + placeholder vision tokens)
D = Qwen text hidden size (e.g., 2048 for Qwen3-VL-2B); must also equal HS encoder output dim
"""

class DualVisionQwen(nn.Module):
    def __init__(self, qwen_model, hs_visual, hs_token_id):
        super().__init__()
        self.qwen = qwen_model
        self.hs_visual = hs_visual    # Stores HS encoder wrapper.
                                        # Expected input/output (from earlier cells):
                                        # Input pixel_values_hs: (B, 285, H, W) (dtype DTYPE)
                                        # Output hs: (B * HS_NUM_TOKENS, D)
        self.hs_token_id = int(hs_token_id)   # integer ID of your special placeholder token <|hs_image_pad|>

    def _base_qwen(self):   # Return the underlying Qwen3VLForConditionalGeneration object even if PEFT wraps it.
        return self.qwen.get_base_model() if hasattr(self.qwen, "get_base_model") else self.qwen

    def forward(self, input_ids, attention_mask=None, labels=None,
                pixel_values=None, image_grid_thw=None,
                pixel_values_hs=None, hs_grid_thw=None):
        """
        Inputs and shapes:
        - input_ids: token IDs for the full prompt, including text + vision placeholder tokens
           Shape: (B, S), dtype long
        - attention_mask (optional): which tokens are “real”
           Shape: (B, S), dtype long/bool depending on tokenizer
        - labels (optional): next-token targets (usually copy of input_ids with masking rules)
           Shape: (B, S), dtype long
        - pixel_values (optional): RGB preprocessed image batch for Qwen’s image tower
           Typical shape: (B, 3, H’, W’) after processor.image_processor
        - image_grid_thw: Qwen’s image token grid metadata
           Shape: (N_img, 3), dtype long (often N_img=1)
        - pixel_values_hs (optional): HS cube batch
           Shape: (B, 285, H, W), dtype DTYPE
        - hs_grid_thw: synthetic HS “image grid” you built earlier
           Shape: (N_hs, 3) (typically (1, 3)), dtype long
        
        """

        ## Load variables onto GPU
        device = next(self.qwen.parameters()).device
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device) if attention_mask is not None else None
        labels = labels.to(device) if labels is not None else None

        base = self._base_qwen()  # Qwen3VLForConditionalGeneration

        ## Grab the embedding vectors from qwen, which are currently placeholders/random values, 
        #     so we can populate them with our vision features:
        inputs_embeds = base.get_input_embeddings()(input_ids)

        assert pixel_values is not None, "RGB image is required"
        assert pixel_values_hs is not None, "HS image is required"

        ## Inject RGB image features into the embedding table we extracted:
        rgb_list, _ = base.get_image_features(pixel_values.to(device), image_grid_thw.to(device))  # Run Qwen vision tower on the RGB image
        rgb = torch.cat(rgb_list, dim=0).to(inputs_embeds.dtype)  # flatten into a list of tokens for injection
        m = (input_ids == base.config.image_token_id).unsqueeze(-1).expand_as(inputs_embeds)   # select the embedding slots that correspond to <|image_pad|> tokens
        inputs_embeds = inputs_embeds.masked_scatter(m, rgb)  # overwrite dummy token embeddings with real ones from Qwen's vision tower

        ## Inject HS image features into the embedding table we extracted:
        hs, _ = self.hs_visual(pixel_values_hs.to(device), grid_thw=hs_grid_thw.to(device))
        hs = hs.to(inputs_embeds.dtype)
        m = (input_ids == self.hs_token_id).unsqueeze(-1).expand_as(inputs_embeds)
        inputs_embeds = inputs_embeds.masked_scatter(m, hs)


        ## This is RoPE (positional indexing) bookkeeping: it makes Qwen’s internal “vision-aware position id builder” treat 
        #     your HS placeholders as if they were normal image tokens, and it supplies grid metadata for both the RGB and HS “images”.
        rope_input_ids = input_ids.clone()
        rope_input_ids[rope_input_ids == self.hs_token_id] = base.config.image_token_id
        
        ## Combine the RGB and HS grid metadata:
        combined_grid = torch.cat([image_grid_thw.to(device), hs_grid_thw.to(device)], dim=0)

        # Call Qwen's RoPE indexer to recompute position IDs for our new token sequence:
        position_ids, _ = base.model.get_rope_index(
            input_ids=rope_input_ids,
            image_grid_thw=combined_grid,
            video_grid_thw=None,
            attention_mask=attention_mask,
        )

        # run Qwen’s text transformer on your already-injected embedding sequence
        lm = base.language_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            position_ids=position_ids,
            use_cache=False,
        )
        
        # turn the resulting hidden states into vocabulary logits.
        logits = base.lm_head(lm.last_hidden_state)

        # Run the result through Qwen's built in loss function to compute a loss if labels are provided. (eg, during training)
        loss = None
        if labels is not None:
            loss = base.loss_function(logits=logits, labels=labels, vocab_size=base.config.text_config.vocab_size)

        return {"loss": loss, "logits": logits}

dual = DualVisionQwen(qwen, hs_encoder, HS_TOKEN_ID).to(DEVICE)

In [9]:
"""
Utilities:
  - num_rgb_tokens: compute how many <|image_pad|> placeholders are needed
  - build_dual_prompt: create RGB vision block + HS vision block + question
  - prep_pseudo_rgb_from_hs: robustly scale selected HS bands to [0,1]
  - prep_hs_norm: bandwise normalize full cube using dataset mean/std
"""

def num_rgb_tokens(image_grid_thw):
    """ compute how many <|image_pad|> placeholders are needed """
    return int((image_grid_thw[0].prod() // (ROPE_MERGE_SIZE**2)).item())

def build_dual_prompt(question, n_rgb, n_hs):
    """ build_dual_prompt: create RGB vision block + HS vision block + question """
    vs, ve = processor.vision_start_token, processor.vision_end_token

    rgb_tokens = " ".join([processor.image_token] * n_rgb)
    hs_tokens  = " ".join([HS_TOKEN] * n_hs)

    rgb_block = f"{vs} {rgb_tokens} {ve}"
    hs_block  = f"{vs} {hs_tokens} {ve}"

    return f"RGB: {rgb_block}\nHS: {hs_block}\n{question}"

def prep_pseudo_rgb_from_hs(hs_raw_chw):
    """ Subsample the HS cube to RGB bands and normalize to [0,1] to generate RGB-Like image """
    rgb = torch.stack([hs_raw_chw[R_BAND], hs_raw_chw[G_BAND], hs_raw_chw[B_BAND]], 0)
    m = torch.isfinite(rgb) & (rgb != FILL)
    rgb = torch.where(m, rgb, torch.zeros_like(rgb))
    v = rgb[m]
    lo, hi = torch.quantile(v, torch.tensor([0.01, 0.99], device=v.device))
    return ((rgb - lo) / (hi - lo + 1e-6)).clamp(0, 1)

def prep_hs_norm(hs_raw_chw, band_mean, band_std):
    """ Normalize HS cube using dataset mean/std """
    # TODO: Verify this method matches with normalization code in the HS autoencoder training notebook
    m = torch.isfinite(hs_raw_chw) & (hs_raw_chw != FILL)
    mean = torch.tensor(band_mean, device=hs_raw_chw.device).view(-1,1,1)
    std  = torch.tensor(band_std,  device=hs_raw_chw.device).view(-1,1,1)
    return torch.where(m, (hs_raw_chw - mean) / std, torch.zeros_like(hs_raw_chw))

def load_hyperspectral_cube_chw_from_npy(hs_path: str, device: str | torch.device | None = None) -> torch.Tensor:
    """
    Loads a hyperspectral cube saved as (H, W, C) numpy and returns a torch tensor (C, H, W).
    Output dtype: float32.
    """
    hs_hwc_float32 = np.load(hs_path).astype(np.float32)            # (H, W, 285)
    hs_chw_float32 = torch.from_numpy(hs_hwc_float32).permute(2, 0, 1)  # (285, H, W)
    return hs_chw_float32.to(device) if device is not None else hs_chw_float32


@torch.no_grad()
def make_rgb_inputs(
    rgb01_chw: torch.Tensor,
    *,
    processor,
    device: str | torch.device,
    dtype: torch.dtype,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Takes a single pseudo-RGB image in [0,1], shape (3, H, W), and returns Qwen-ready RGB inputs.
    Returns:
      - rgb_pixel_values: (1, 3, H', W')  (after processor resizing/normalization)
      - image_grid_thw:   (1, 3)
    """
    rgb_inputs = processor.image_processor(images=rgb01_chw, return_tensors="pt")
    rgb_pixel_values = rgb_inputs["pixel_values"].to(device=device, dtype=dtype)
    image_grid_thw = rgb_inputs["image_grid_thw"].to(device=device)
    return rgb_pixel_values, image_grid_thw


@torch.no_grad()
def prepare_hs_pixel_values_batch(
    hs_chw_float32: torch.Tensor,
    *,
    band_mean,
    band_std,
    device: str | torch.device,
    dtype: torch.dtype,
) -> torch.Tensor:
    """
    Normalizes a single HS cube (285, H, W) and returns a batch (1, 285, H, W) suitable for hs_encoder/dual().
    """
    hs_norm_chw = prep_hs_norm(hs_chw_float32, band_mean, band_std)          # (285, H, W)
    hs_pixel_values = hs_norm_chw.unsqueeze(0).to(device=device, dtype=dtype)  # (1, 285, H, W)
    return hs_pixel_values


def build_chat_prompt_text_for_inference(
    *,
    processor,
    user_content: str,
    system_prompt: str = "You are a helpful assistant. Answer concisely.",
) -> str:
    """
    Builds the final text prompt for Qwen Instruct models (chat template + assistant generation turn).
    """
    chat_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]
    prompt_text = processor.tokenizer.apply_chat_template(
        chat_messages,
        tokenize=False,
        add_generation_prompt=True,   # critical for Instruct models to actually generate an answer
    )
    return prompt_text


@torch.no_grad()
def greedy_decode_one_sample(
    *,
    dual_model,
    input_ids: torch.Tensor,               # (1, S)
    attention_mask: torch.Tensor | None,   # (1, S) or None
    max_new_tokens: int,
    eos_token_id: int,
    dual_static_kwargs: dict,              # pixel_values/image_grid_thw/pixel_values_hs/hs_grid_thw
) -> torch.Tensor:
    """
    Greedy decode loop for a single sample (batch size 1).
    Returns the full token sequence (prompt + generated).
    """
    for _ in range(max_new_tokens):
        dual_out = dual_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **dual_static_kwargs,
        )
        next_token_id = dual_out["logits"][:, -1].argmax(dim=-1, keepdim=True)  # (1, 1)

        input_ids = torch.cat([input_ids, next_token_id], dim=1)               # (1, S+1)
        if attention_mask is not None:
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token_id)], dim=1)

        if int(next_token_id.item()) == int(eos_token_id):
            break

    return input_ids


In [10]:
band_mean, band_std, _ = load_band_stats(BAND_STATS_PATH)

class PlumeExistsDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths
        self.labels = labels

    def __len__(self): 
        return len(self.paths)

    def __getitem__(self, i):
        y = int(self.labels[i])
        hs_hwc = np.load(self.paths[i]).astype(np.float32)                  # (256,256,285)
        hs = torch.from_numpy(hs_hwc).permute(2,0,1)                        # (285,256,256)
        rgb01 = prep_pseudo_rgb_from_hs(hs)
        hs_norm = prep_hs_norm(hs, band_mean, band_std)
        return rgb01, hs_norm, y

# Custom collate function to assemble batches of combined RGB, HS, and label tensors
def collate_fn(batch):
    rgbs, hss, ys = zip(*batch)
    rgbs = torch.stack(rgbs, 0)          # (B,3,256,256)
    hss  = torch.stack(hss, 0)           # (B,285,256,256)
    ys   = torch.tensor(ys, dtype=torch.long)
    return rgbs, hss, ys

In [11]:
"""
Cell: build plume/negative dataset from path conventions

Positive (plume) cubes:
  /more_data/emit_dataset_2/*/CH4_*/hypercube.npy

Negative cubes:
  /more_data/emit_dataset_2/*/negative_*/hypercube.npy
"""

POS_GLOB = "/more_data/emit_dataset_2/*/CH4_*/hypercube.npy"
NEG_GLOB = "/more_data/emit_dataset_2/*/negative_*/hypercube.npy"

pos_paths = sorted(glob.glob(POS_GLOB))
neg_paths = sorted(glob.glob(NEG_GLOB))

random.seed(0)
random.shuffle(pos_paths)
random.shuffle(neg_paths)

# balance classes by downsampling negatives - should already be done in dataset gen code but just in case
n = min(len(pos_paths), len(neg_paths))
pos_paths, neg_paths = pos_paths[:n], neg_paths[:n]

all_paths = pos_paths + neg_paths
all_labels = [1]*len(pos_paths) + [0]*len(neg_paths)

# Reshuffle data pairs now that they're combined
paired = list(zip(all_paths, all_labels))
random.seed(0)
random.shuffle(paired)
all_paths, all_labels = zip(*paired)
all_paths, all_labels = list(all_paths), list(all_labels)

# train/val split
split = int(0.9 * len(all_paths))
train_paths, val_paths = all_paths[:split], all_paths[split:]
train_labels, val_labels = all_labels[:split], all_labels[split:]

train_ds = PlumeExistsDataset(train_paths, train_labels)
val_ds   = PlumeExistsDataset(val_paths, val_labels)

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    num_workers=0,        
    pin_memory=False,    
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_fn
)

len(train_ds), len(val_ds), len(pos_paths), len(neg_paths)

(802, 90, 446, 446)

In [12]:
## Test blind inference
@torch.no_grad()
def generate_dual_answer_from_hs_path(
    *,
    question: str,
    hs_path: str,
    dual_model,
    processor,
    hs_grid_thw: torch.Tensor,
    band_mean,
    band_std,
    device: str | torch.device = DEVICE,
    dtype: torch.dtype = DTYPE,
    max_new_tokens: int = 64,
    system_prompt: str = "You are a helpful assistant. Answer concisely.",
) -> str:
    """
    End-to-end single-sample inference:
      HS npy -> pseudo-RGB + normalized HS -> dual prompt -> greedy decode -> decoded assistant text
    """
    # 1) Load HS cube (C, H, W)
    hs_chw_float32 = load_hyperspectral_cube_chw_from_npy(hs_path, device=device)  # (285, H, W)

    # 2) Prepare RGB + HS model inputs (both derived from the same HS cube)
    rgb01_chw = prep_pseudo_rgb_from_hs(hs_chw_float32)  # (3, H, W), values in [0, 1]
    rgb_pixel_values, image_grid_thw = make_rgb_inputs(
        rgb01_chw,
        processor=processor,
        device=device,
        dtype=dtype,
    )
    hs_pixel_values = prepare_hs_pixel_values_batch(
        hs_chw_float32,
        band_mean=band_mean,
        band_std=band_std,
        device=device,
        dtype=dtype,
    )

    # 3) Build dual-vision “user content” and wrap in chat template
    num_image_placeholders = num_rgb_tokens(image_grid_thw)
    user_content = build_dual_prompt(question, num_image_placeholders, HS_NUM_TOKENS)
    prompt_text = build_chat_prompt_text_for_inference(
        processor=processor,
        user_content=user_content,
        system_prompt=system_prompt,
    )

    # 4) Tokenize
    tokenized = processor.tokenizer(prompt_text, return_tensors="pt").to(device)
    input_ids = tokenized["input_ids"]                       # (1, S)
    attention_mask = tokenized.get("attention_mask", None)   # (1, S) or None
    prompt_token_count = input_ids.shape[1]

    # 5) Greedy decode using the dual model
    dual_static_kwargs = {
        "pixel_values": rgb_pixel_values,
        "image_grid_thw": image_grid_thw,
        "pixel_values_hs": hs_pixel_values,
        "hs_grid_thw": hs_grid_thw.to(device),
    }
    full_ids = greedy_decode_one_sample(
        dual_model=dual_model,
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        eos_token_id=processor.tokenizer.eos_token_id,
        dual_static_kwargs=dual_static_kwargs,
    )

    # 6) Decode only the generated assistant continuation (not the prompt)
    generated_ids = full_ids[0, prompt_token_count:]
    generated_text = processor.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return generated_text

HS_PATH = "/more_data/emit_dataset_2/20230809T060151/CH4_PlumeComplex-1371_huge_plume/hypercube.npy"

answer_text = generate_dual_answer_from_hs_path(
    question="Is there a plume in this image? Answer yes or no.",
    hs_path=HS_PATH,
    dual_model=dual,                # DualVisionQwen(...)
    processor=processor,            # AutoProcessor.from_pretrained(...)
    hs_grid_thw=hs_grid_thw,        # synthetic grid tensor created earlier
    band_mean=band_mean,            # from load_band_stats(...) or ckpt
    band_std=band_std,              # from load_band_stats(...) or ckpt
    device=DEVICE,
    dtype=DTYPE,
    max_new_tokens=16,
    system_prompt="You are a helpful assistant. Answer with exactly one word: yes or no.",
)

print(answer_text)

Yes, the 100000000000


In [13]:


def train_dual_qwen_hsi_with_lora(
    *,
    qwen,
    hs_encoder,
    hs_token_id: int,
    hs_grid_thw: torch.Tensor,
    processor,
    train_loader,
    device: str | torch.device,
    dtype: torch.dtype,
    question_text: str = "Is there a plume in this image?",
    system_prompt: str = "Answer with exactly one word: yes or no.",
    lora_r: int = 8,
    lora_alpha: int = 16,
    lora_dropout: float = 0.05,
    lora_target_modules: list[str] = ["q_proj", "k_proj", "v_proj", "o_proj"],
    learning_rate: float = 1e-4,
    epochs: int = 1,
    max_steps_per_epoch: int | None = 200,
    grad_clip_norm: float | None = 1.0,
):
    """
    Minimal LoRA finetune for the dual RGB+HS Qwen wrapper.

    Assumptions (matches your notebook):
      - `DualVisionQwen`, `build_dual_prompt`, `num_rgb_tokens`, `HS_NUM_TOKENS` are already defined.
      - `train_loader` yields (rgb01_b3hw, hs_norm_bchw, class_labels_b)
      - Current implementation assumes batch_size == 1 (same as your DataLoader setup).
    """
    yes_text, no_text = "yes", "no"

    def tokenize_supervised_yesno_example(*, class_label: int, image_grid_thw: torch.Tensor):
        assistant_answer_text = yes_text if int(class_label) == 1 else no_text
        user_content = build_dual_prompt(question_text, num_rgb_tokens(image_grid_thw), HS_NUM_TOKENS)

        full_messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_answer_text},
        ]
        full_text = processor.tokenizer.apply_chat_template(
            full_messages, tokenize=False, add_generation_prompt=False
        )
        tokenized_full = processor.tokenizer(full_text, return_tensors="pt")

        prompt_only_messages = full_messages[:-1]
        prompt_only_text = processor.tokenizer.apply_chat_template(
            prompt_only_messages, tokenize=False, add_generation_prompt=True
        )
        tokenized_prompt_only = processor.tokenizer(prompt_only_text, return_tensors="pt")

        labels = tokenized_full["input_ids"].clone()
        prompt_token_count = tokenized_prompt_only["input_ids"].shape[1]
        labels[:, :prompt_token_count] = -100
        return tokenized_full, labels

    # Freeze Qwen base weights
    for param in qwen.parameters():
        param.requires_grad = False

    # Keep HS encoder trainable
    for param in hs_encoder.parameters():
        param.requires_grad = True

    # Apply LoRA to full Qwen model (PEFT expects prepare_inputs_for_generation on the full model)
    if not hasattr(qwen, "peft_config"):
        lora_cfg = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=lora_r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=lora_target_modules,
            bias="none",
        )
        qwen = get_peft_model(qwen, lora_cfg)
        qwen.print_trainable_parameters()

    dual_model = DualVisionQwen(qwen, hs_encoder, hs_token_id).to(device)

    trainable_parameters = [p for p in hs_encoder.parameters() if p.requires_grad] + \
                           [p for p in qwen.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_parameters, lr=learning_rate)

    dual_model.train()
    for epoch_index in range(epochs):
        running_loss = 0.0
        step_count = 0

        for step_index, (rgb01_b3hw, hs_norm_bchw, class_labels_b) in enumerate(train_loader):
            if max_steps_per_epoch is not None and step_index >= max_steps_per_epoch:
                break

            assert rgb01_b3hw.shape[0] == 1, "This minimal loop assumes batch_size=1."

            rgb01_b3hw = rgb01_b3hw.to(device)
            hs_norm_bchw = hs_norm_bchw.to(device=device, dtype=dtype)

            # Qwen RGB vision-tower inputs (keep it per-sample, simple, and consistent with your notebook)
            rgb_inputs = processor.image_processor(images=rgb01_b3hw[0], return_tensors="pt")
            rgb_pixel_values = rgb_inputs["pixel_values"].to(device=device, dtype=dtype)
            image_grid_thw = rgb_inputs["image_grid_thw"].to(device=device)

            tokenized_full, labels = tokenize_supervised_yesno_example(
                class_label=int(class_labels_b[0].item()),
                image_grid_thw=image_grid_thw,
            )
            input_ids = tokenized_full["input_ids"].to(device)
            attention_mask = tokenized_full.get("attention_mask", None)
            attention_mask = attention_mask.to(device) if attention_mask is not None else None
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)

            out = dual_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                pixel_values=rgb_pixel_values,
                image_grid_thw=image_grid_thw,
                pixel_values_hs=hs_norm_bchw,
                hs_grid_thw=hs_grid_thw,
            )

            loss = out["loss"]
            loss.backward()

            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(trainable_parameters, grad_clip_norm)

            optimizer.step()

            running_loss += float(loss)
            step_count += 1

        print(f"epoch {epoch_index} loss {running_loss / max(1, step_count):.4f}")

    return dual_model


# --- run it ---
dual_model = train_dual_qwen_hsi_with_lora(
    qwen=qwen,
    hs_encoder=hs_encoder,
    hs_token_id=HS_TOKEN_ID,
    hs_grid_thw=hs_grid_thw,
    processor=processor,
    train_loader=train_loader,
    device=DEVICE,
    dtype=DTYPE,
    epochs=10,
    max_steps_per_epoch=64,
)

trainable params: 3,211,264 || all params: 2,130,198,528 || trainable%: 0.1507
epoch 0 loss 0.6883
epoch 1 loss 0.3201
epoch 2 loss 0.3266
epoch 3 loss 0.3235
epoch 4 loss 0.2526
epoch 5 loss 0.2640
epoch 6 loss 0.2464
epoch 7 loss 0.2369
epoch 8 loss 0.2384
epoch 9 loss 0.2348


In [14]:
import torch

@torch.no_grad()
def compute_plume_yesno_confusion_matrix(
    *,
    dual_model,
    processor,
    val_loader,
    hs_grid_thw: torch.Tensor,
    device: str | torch.device,
    dtype: torch.dtype,
    question_text: str = "Is there a plume in this image? Answer yes or no.",
    system_prompt: str = "Answer with exactly one word: yes or no.",
    max_new_tokens: int = 8,
):
    """
    Runs greedy decoding over the full val_loader and computes TP/FN/FP/TN.
    Assumes val_loader yields (rgb01_b3hw, hs_norm_bchw, class_labels_b) with batch_size == 1.
    Uses existing notebook helpers: build_dual_prompt, num_rgb_tokens, HS_NUM_TOKENS,
    build_chat_prompt_text_for_inference, greedy_decode_one_sample.
    """
    def parse_yesno_to_label(generated_text: str) -> int | None:
        t = (generated_text or "").strip().lower()
        if t.startswith("yes"):
            return 1
        if t.startswith("no"):
            return 0
        return None

    dual_model.eval()

    tp = fn = fp = tn = 0
    unknown = 0

    for rgb01_b3hw, hs_norm_bchw, class_labels_b in val_loader:
        assert rgb01_b3hw.shape[0] == 1, "This minimal eval loop assumes batch_size=1."
        y_true = int(class_labels_b[0].item())

        rgb01_b3hw = rgb01_b3hw.to(device)
        hs_norm_bchw = hs_norm_bchw.to(device=device, dtype=dtype)

        rgb_inputs = processor.image_processor(images=rgb01_b3hw[0], return_tensors="pt")
        rgb_pixel_values = rgb_inputs["pixel_values"].to(device=device, dtype=dtype)
        image_grid_thw = rgb_inputs["image_grid_thw"].to(device=device)

        user_content = build_dual_prompt(question_text, num_rgb_tokens(image_grid_thw), HS_NUM_TOKENS)
        prompt_text = build_chat_prompt_text_for_inference(
            processor=processor,
            user_content=user_content,
            system_prompt=system_prompt,
        )

        tokenized = processor.tokenizer(prompt_text, return_tensors="pt").to(device)
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized.get("attention_mask", None)
        prompt_token_count = input_ids.shape[1]

        dual_static_kwargs = {
            "pixel_values": rgb_pixel_values,
            "image_grid_thw": image_grid_thw,
            "pixel_values_hs": hs_norm_bchw,
            "hs_grid_thw": hs_grid_thw.to(device),
        }

        full_ids = greedy_decode_one_sample(
            dual_model=dual_model,
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            eos_token_id=processor.tokenizer.eos_token_id,
            dual_static_kwargs=dual_static_kwargs,
        )

        generated_ids = full_ids[0, prompt_token_count:]
        generated_text = processor.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        y_pred = parse_yesno_to_label(generated_text)
        if y_pred is None:
            unknown += 1
            continue

        if y_true == 1 and y_pred == 1:
            tp += 1
        elif y_true == 1 and y_pred == 0:
            fn += 1
        elif y_true == 0 and y_pred == 1:
            fp += 1
        elif y_true == 0 and y_pred == 0:
            tn += 1

    return {"TP": tp, "FN": fn, "FP": fp, "TN": tn, "unknown": unknown}


cm = compute_plume_yesno_confusion_matrix(
    dual_model=dual_model,
    processor=processor,
    val_loader=val_loader,
    hs_grid_thw=hs_grid_thw,
    device=DEVICE,
    dtype=DTYPE,
)

print(cm)

{'TP': 2, 'FN': 43, 'FP': 3, 'TN': 42, 'unknown': 0}


In [15]:
# ## Train Combined Model:
# #Constants:
# lora_r = 8
# lora_alpha = 16
# lora_dropout = 0.05
# lora_target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
# learning_rate = 1e-4

# # Labels:
# YES = "yes"
# NO  = "no"

# # Freeze everything first
# for p in qwen.parameters():
#     p.requires_grad = False

# # Keep HS encoder trainable
# for p in hs_encoder.parameters():
#     p.requires_grad = True

# # Apply LoRA to the *full* Qwen model, but only on LM attention projections
# lora_cfg = LoraConfig(
#     task_type=TaskType.CAUSAL_LM,
#     r=lora_r,
#     lora_alpha=lora_alpha,
#     lora_dropout=lora_dropout,
#     target_modules=lora_target_modules,
#     bias="none",
# )

# qwen = get_peft_model(qwen, lora_cfg)
# qwen.print_trainable_parameters()

# model = DualVisionQwen(qwen, hs_encoder, HS_TOKEN_ID).to(DEVICE)  # load the combined model

# # Grab HS encoder + LoRA trainable params for the optimizer
# train_params = [p for p in hs_encoder.parameters() if p.requires_grad] + \
#                [p for p in qwen.parameters() if p.requires_grad]
# opt = torch.optim.AdamW(train_params, lr=learning_rate)


# def train_one_epoch(loader, max_steps=None):
#     dual.train()
#     total = 0.0
#     for step, (rgb01, hs_norm, y) in enumerate(loader):
#         if max_steps is not None and step >= max_steps: break

#         rgb01 = rgb01.to(DEVICE)
#         hs_norm = hs_norm.to(DEVICE, dtype=DTYPE)

#         rgb_pixel_values, image_grid_thw = make_rgb_inputs(rgb01)
#         rgb_pixel_values = rgb_pixel_values.to(DEVICE, dtype=DTYPE)
#         image_grid_thw = image_grid_thw.to(DEVICE)

#         tok = build_example("Is there a plume in this image?", y.item(), image_grid_thw)
#         input_ids = tok["input_ids"].to(DEVICE)
#         attention_mask = tok.get("attention_mask", None)
#         attention_mask = attention_mask.to(DEVICE) if attention_mask is not None else None
#         labels = input_ids.clone()

#         out = dual(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             labels=labels,
#             pixel_values=rgb_pixel_values,
#             image_grid_thw=image_grid_thw,
#             pixel_values_hs=hs_norm,
#             hs_grid_thw=hs_grid_thw,
#         )

#         opt.zero_grad(set_to_none=True)
#         out["loss"].backward()
#         opt.step()
#         total += float(out["loss"])
#     return total / max(1, step+1)


In [16]:
# # --- existing imports assumed above this cell ---
# # from peft import LoraConfig, get_peft_model, TaskType

# # --- existing config assumed above this cell ---
# # lora_r = 8
# # lora_alpha = 16
# # lora_dropout = 0.05
# # lora_target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
# # learning_rate = 1e-4
# # YES = "yes"
# # NO = "no"

# # -----------------------------
# # LoRA + optimizer setup
# # -----------------------------

# # Freeze everything first
# for param in qwen.parameters():
#     param.requires_grad = False

# # Keep HS encoder trainable
# for param in hs_encoder.parameters():
#     param.requires_grad = True

# lora_cfg = LoraConfig(
#     task_type=TaskType.CAUSAL_LM,
#     r=lora_r,
#     lora_alpha=lora_alpha,
#     lora_dropout=lora_dropout,
#     target_modules=lora_target_modules,
#     bias="none",
# )

# # Wrap the full Qwen model with LoRA adapters
# qwen = get_peft_model(qwen, lora_cfg)
# qwen.print_trainable_parameters()

# # IMPORTANT: rebuild the dual wrapper to use the LoRA-wrapped qwen
# model = DualVisionQwen(qwen, hs_encoder, HS_TOKEN_ID).to(DEVICE)

# # Optimizer over HS encoder + LoRA params (base Qwen weights remain frozen)
# trainable_parameters = [p for p in hs_encoder.parameters() if p.requires_grad] + \
#                        [p for p in qwen.parameters() if p.requires_grad]
# opt = torch.optim.AdamW(trainable_parameters, lr=learning_rate)


# # -----------------------------
# # Small helpers (shared structure with inference)
# # -----------------------------

# @torch.no_grad()
# def make_rgb_inputs(rgb01_b3hw: torch.Tensor):
#     """
#     Converts a batch of pseudo-RGB images in [0,1] into Qwen vision-tower inputs.

#     rgb01_b3hw: (B, 3, H, W)
#     returns:
#       pixel_values:   (B, 3, H', W')
#       image_grid_thw: (B, 3)   (or sometimes (N_img, 3) depending on processor)
#     """
#     rgb_images = [rgb01_b3hw[i] for i in range(rgb01_b3hw.shape[0])]  # list of (3, H, W)
#     rgb_inputs = processor.image_processor(images=rgb_images, return_tensors="pt")
#     return rgb_inputs["pixel_values"], rgb_inputs["image_grid_thw"]


# def build_supervised_yesno_tokens(
#     *,
#     question_text: str,
#     class_label: int,
#     image_grid_thw: torch.Tensor,
# ):
#     """
#     Builds a supervised chat example and returns:
#       - tokenized_full: dict with input_ids/attention_mask for the full conversation (includes assistant answer)
#       - labels: (1, S) with prompt tokens masked to -100 so loss trains only on the assistant answer
#     """
#     assistant_answer_text = YES if int(class_label) == 1 else NO

#     # 1) User content contains the dual vision placeholders + question
#     user_content = build_dual_prompt(question_text, num_rgb_tokens(image_grid_thw), HS_NUM_TOKENS)

#     # 2) Full conversation (supervised): includes assistant answer
#     full_messages = [
#         {"role": "system", "content": "Answer with exactly one word: yes or no."},
#         {"role": "user", "content": user_content},
#         {"role": "assistant", "content": assistant_answer_text},
#     ]
#     full_prompt_text = processor.tokenizer.apply_chat_template(
#         full_messages, tokenize=False, add_generation_prompt=False
#     )
#     tokenized_full = processor.tokenizer(full_prompt_text, return_tensors="pt")

#     # 3) Prompt-only conversation: same but WITHOUT assistant answer, and with a generation prompt
#     prompt_only_messages = full_messages[:-1]
#     prompt_only_text = processor.tokenizer.apply_chat_template(
#         prompt_only_messages, tokenize=False, add_generation_prompt=True
#     )
#     tokenized_prompt_only = processor.tokenizer(prompt_only_text, return_tensors="pt")

#     # 4) Labels: ignore prompt, train only on assistant continuation
#     labels = tokenized_full["input_ids"].clone()
#     prompt_token_count = tokenized_prompt_only["input_ids"].shape[1]
#     labels[:, :prompt_token_count] = -100

#     return tokenized_full, labels


# # -----------------------------
# # Train loop
# # -----------------------------

# def train_one_epoch(loader, *, max_steps: int | None = None) -> float:
#     model.train()
#     running_loss = 0.0
#     step_count = 0

#     for step_index, (rgb01_b3hw, hs_norm_bchw, class_labels_b) in enumerate(loader):
#         if max_steps is not None and step_index >= max_steps:
#             break

#         # Move batch to device
#         rgb01_b3hw = rgb01_b3hw.to(DEVICE)                         # (B, 3, H, W)
#         hs_norm_bchw = hs_norm_bchw.to(DEVICE, dtype=DTYPE)        # (B, 285, H, W)

#         # Qwen RGB inputs
#         rgb_pixel_values, image_grid_thw = make_rgb_inputs(rgb01_b3hw)
#         rgb_pixel_values = rgb_pixel_values.to(DEVICE, dtype=DTYPE)
#         image_grid_thw = image_grid_thw.to(DEVICE)

#         # This training code assumes batch_size == 1 for prompt construction (matches your DataLoader).
#         # If you later set batch_size > 1, you should build/tokenize per-sample and pad/stack.
#         assert rgb01_b3hw.shape[0] == 1, "This training loop currently assumes batch_size=1."

#         tokenized_full, labels = build_supervised_yesno_tokens(
#             question_text="Is there a plume in this image?",
#             class_label=int(class_labels_b[0].item()),
#             image_grid_thw=image_grid_thw,
#         )

#         input_ids = tokenized_full["input_ids"].to(DEVICE)  # (1, S)
#         attention_mask = tokenized_full.get("attention_mask", None)
#         attention_mask = attention_mask.to(DEVICE) if attention_mask is not None else None
#         labels = labels.to(DEVICE)

#         out = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             labels=labels,
#             pixel_values=rgb_pixel_values,
#             image_grid_thw=image_grid_thw,
#             pixel_values_hs=hs_norm_bchw,
#             hs_grid_thw=hs_grid_thw,
#         )

#         opt.zero_grad(set_to_none=True)
#         out["loss"].backward()
#         opt.step()

#         running_loss += float(out["loss"])
#         step_count += 1

#     return running_loss / max(1, step_count)


# # -----------------------------
# # Run training
# # -----------------------------

# EPOCHS = 1
# for epoch_index in range(EPOCHS):
#     epoch_loss = train_one_epoch(train_loader, max_steps=200)
#     print(f"epoch {epoch_index} loss {epoch_loss:.4f}")

In [17]:
# ## Cell 6 — Single forward pass smoke test

# band_mean, band_std, _ = load_band_stats(BAND_STATS_PATH)

# HS_PATH = "/more_data/emit_dataset_2/20230809T060151/CH4_PlumeComplex-1371_huge_plume/hypercube.npy"  # set this
# hs_hwc = np.load(HS_PATH).astype(np.float32)                 # (256,256,285)
# hs = torch.from_numpy(hs_hwc).permute(2,0,1).to(DEVICE)      # (285,256,256)

# rgb01 = prep_pseudo_rgb_from_hs(hs)
# rgb_inputs = processor.image_processor(images=rgb01, return_tensors="pt")
# pixel_values = rgb_inputs["pixel_values"].to(DEVICE, dtype=DTYPE)
# image_grid_thw = rgb_inputs["image_grid_thw"].to(DEVICE)

# hs_norm = prep_hs_norm(hs, band_mean, band_std)
# pixel_values_hs = hs_norm.unsqueeze(0).to(DEVICE, dtype=DTYPE)

# question = "Is there a plume in this image? Answer yes or no."
# prompt = build_dual_prompt(question, num_rgb_tokens(image_grid_thw), HS_NUM_TOKENS)
# tok = processor.tokenizer(prompt, return_tensors="pt")

# out = dual(
#     input_ids=tok["input_ids"],
#     attention_mask=tok.get("attention_mask", None),
#     pixel_values=pixel_values,
#     image_grid_thw=image_grid_thw,
#     pixel_values_hs=pixel_values_hs,
#     hs_grid_thw=hs_grid_thw,
# )
# out["logits"].shape

In [18]:
# ## Cell 7 — Minimal finetune step (HS encoder only; Qwen frozen)

# for p in qwen.parameters():
#     p.requires_grad = False

# opt = torch.optim.AdamW([p for p in hs_encoder.parameters() if p.requires_grad], lr=1e-4)

# labels = tok["input_ids"].clone()
# train = dual(
#     input_ids=tok["input_ids"],
#     attention_mask=tok.get("attention_mask", None),
#     labels=labels,
#     pixel_values=pixel_values,
#     image_grid_thw=image_grid_thw,
#     pixel_values_hs=pixel_values_hs,
#     hs_grid_thw=hs_grid_thw,
# )

# opt.zero_grad(set_to_none=True)
# train["loss"].backward()
# opt.step()
# float(train["loss"])

In [19]:
# """
# Cell: inference (single datapoint) — Qwen chat-format prompt + greedy decode

# Why your previous output had no answer:
#   Qwen3-* Instruct models expect a chat template with an explicit "assistant turn".
#   Without that, the model often emits EOS immediately, so decoding shows only the prompt.
# """

# dual.eval(); qwen.eval(); hs_encoder.eval()

# @torch.no_grad()
# def generate_dual(question, hs_path, band_stats_path="emit_band_stats.npz", max_new_tokens=64):
#     band_mean, band_std, _ = load_band_stats(band_stats_path)

#     hs_hwc = np.load(hs_path).astype(np.float32)                 # (256,256,285)
#     hs = torch.from_numpy(hs_hwc).permute(2,0,1).to(DEVICE)      # (285,256,256)

#     rgb01 = prep_pseudo_rgb_from_hs(hs)
#     rgb_inputs = processor.image_processor(images=rgb01, return_tensors="pt")
#     pixel_values = rgb_inputs["pixel_values"].to(DEVICE, dtype=DTYPE)
#     image_grid_thw = rgb_inputs["image_grid_thw"].to(DEVICE)

#     hs_norm = prep_hs_norm(hs, band_mean, band_std)
#     pixel_values_hs = hs_norm.unsqueeze(0).to(DEVICE, dtype=DTYPE)

#     # Build the dual-vision "user content" (contains the two vision blocks + question)
#     user_content = build_dual_prompt(question, num_rgb_tokens(image_grid_thw), HS_NUM_TOKENS)

#     # Wrap in chat format and add an explicit assistant generation prompt
#     messages = [
#         {"role": "system", "content": "You are a helpful assistant. Answer concisely."},
#         {"role": "user", "content": user_content},
#     ]
#     prompt = processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

#     tok = processor.tokenizer(prompt, return_tensors="pt").to(DEVICE)
#     input_ids = tok["input_ids"]
#     attention_mask = tok.get("attention_mask", None)

#     prompt_len = input_ids.shape[1]
#     eos_id = processor.tokenizer.eos_token_id

#     for _ in range(max_new_tokens):
#         out = dual(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             pixel_values=pixel_values,
#             image_grid_thw=image_grid_thw,
#             pixel_values_hs=pixel_values_hs,
#             hs_grid_thw=hs_grid_thw,
#         )
#         next_id = out["logits"][:, -1].argmax(dim=-1, keepdim=True)
#         input_ids = torch.cat([input_ids, next_id], dim=1)
#         if attention_mask is not None:
#             attention_mask = torch.cat([attention_mask, torch.ones_like(next_id)], dim=1)
#         if int(next_id.item()) == int(eos_id):
#             break

#     # Return ONLY the newly generated assistant tokens (not the prompt)
#     gen_ids = input_ids[0, prompt_len:]
#     return processor.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

# # Example
# HS_PATH = "/more_data/emit_dataset_2/20230809T060151/CH4_PlumeComplex-1371_huge_plume/hypercube.npy"
# print(generate_dual("Is there a plume in this image? Answer yes or no.", HS_PATH, max_new_tokens=16))

In [20]:
# """
# Cell: LoRA for Qwen3-VL (apply to full model; target LM attention proj only)

# Why this way:
#   PEFT's CAUSAL_LM wrapper expects prepare_inputs_for_generation().
#   Qwen3VLForConditionalGeneration has it; qwen.language_model (Qwen3VLTextModel) does not.
# """

# from peft import LoraConfig, get_peft_model, TaskType

# # Freeze everything first
# for p in qwen.parameters():
#     p.requires_grad = False

# # Keep HS encoder trainable
# for p in hs_encoder.parameters():
#     p.requires_grad = True

# # Apply LoRA to the *full* Qwen model, but only on LM attention projections
# lora_cfg = LoraConfig(
#     task_type=TaskType.CAUSAL_LM,
#     r=8,
#     lora_alpha=16,
#     lora_dropout=0.05,
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
#     bias="none",
# )

# qwen = get_peft_model(qwen, lora_cfg)
# qwen.print_trainable_parameters()

# # Rebuild dual wrapper so it uses the LoRA-wrapped qwen
# dual = DualVisionQwen(qwen, hs_encoder, HS_TOKEN_ID).to(DEVICE)

# # Optimizer over HS encoder + LoRA params
# train_params = [p for p in hs_encoder.parameters() if p.requires_grad] + \
#                [p for p in qwen.parameters() if p.requires_grad]
# opt = torch.optim.AdamW(train_params, lr=1e-4)

# YES = "yes"
# NO  = "no"

# @torch.no_grad()
# def make_rgb_inputs(rgb01_b3hw):
#     # processor expects images per-sample; loop keeps this simple for B=1
#     out = processor.image_processor(images=rgb01_b3hw[0], return_tensors="pt")
#     return out["pixel_values"], out["image_grid_thw"]

# def build_example(question, y, image_grid_thw):
#     # user content includes dual vision blocks; wrap in chat template for instruct behavior
#     user_content = build_dual_prompt(question, num_rgb_tokens(image_grid_thw), HS_NUM_TOKENS)
#     messages = [
#         {"role": "system", "content": "Answer with exactly one word: yes or no."},
#         {"role": "user", "content": user_content},
#         {"role": "assistant", "content": YES if int(y)==1 else NO},
#     ]
#     prompt = processor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
#     tok = processor.tokenizer(prompt, return_tensors="pt")
#     return tok

# def train_one_epoch(loader, max_steps=None):
#     dual.train()
#     total = 0.0
#     for step, (rgb01, hs_norm, y) in enumerate(loader):
#         if max_steps is not None and step >= max_steps: break

#         rgb01 = rgb01.to(DEVICE)
#         hs_norm = hs_norm.to(DEVICE, dtype=DTYPE)

#         rgb_pixel_values, image_grid_thw = make_rgb_inputs(rgb01)
#         rgb_pixel_values = rgb_pixel_values.to(DEVICE, dtype=DTYPE)
#         image_grid_thw = image_grid_thw.to(DEVICE)

#         tok = build_example("Is there a plume in this image?", y.item(), image_grid_thw)
#         input_ids = tok["input_ids"].to(DEVICE)
#         attention_mask = tok.get("attention_mask", None)
#         attention_mask = attention_mask.to(DEVICE) if attention_mask is not None else None
#         labels = input_ids.clone()

#         out = dual(
#             input_ids=input_ids,
#             attention_mask=attention_mask,
#             labels=labels,
#             pixel_values=rgb_pixel_values,
#             image_grid_thw=image_grid_thw,
#             pixel_values_hs=hs_norm,
#             hs_grid_thw=hs_grid_thw,
#         )

#         opt.zero_grad(set_to_none=True)
#         out["loss"].backward()
#         opt.step()
#         total += float(out["loss"])
#     return total / max(1, step+1)

# # run a short fine-tune
# EPOCHS = 1
# for e in range(EPOCHS):
#     loss = train_one_epoch(train_loader, max_steps=200)  # cap while iterating
#     print("epoch", e, "loss", loss)

In [21]:
# import numpy as np
# import glob
# import random
# import gc

# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# from transformers import AutoConfig, AutoProcessor, Qwen3VLForConditionalGeneration


In [22]:
# ## Merge our HSI encoder with Qwen3-VL

# qwen_name = "Qwen/Qwen3-VL-2B-Instruct"
# processor = AutoProcessor.from_pretrained(qwen_name)
# qwen = Qwen3VLForConditionalGeneration.from_pretrained(qwen_name, torch_dtype=torch.bfloat16, device_map="auto")

# HS_TOKEN = "<|hs_image_pad|>"  # new token string

# num_added = processor.tokenizer.add_special_tokens({"additional_special_tokens": [HS_TOKEN]})
# if num_added > 0:
#     qwen.resize_token_embeddings(len(processor.tokenizer))

# HS_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids(HS_TOKEN)
# print("HS_TOKEN_ID:", HS_TOKEN_ID, "added:", num_added)

In [23]:
# class DualVisionQwen(nn.Module):
#     def __init__(self, qwen_model, hs_visual, hs_token_id):
#         super().__init__()
#         self.qwen = qwen_model                 # Qwen3VLForConditionalGeneration
#         self.hs_visual = hs_visual             # your HyperspectralVisionEncoder (NOT autoencoder)
#         self.hs_token_id = int(hs_token_id)    # token id for <|hs_image_pad|>

#         # sanity: HS encoder output dim must match Qwen text hidden size
#         assert self.hs_visual.hidden_size == self.qwen.config.text_config.hidden_size

#     def forward(
#         self,
#         input_ids,
#         attention_mask=None,
#         labels=None,
#         pixel_values=None,
#         image_grid_thw=None,
#         pixel_values_hs=None,
#         hs_grid_thw=None,
#         **kwargs,
#     ):
#         device = self.qwen.device if hasattr(self.qwen, "device") else input_ids.device
#         input_ids = input_ids.to(device)
#         if attention_mask is not None:
#             attention_mask = attention_mask.to(device)
#         if labels is not None:
#             labels = labels.to(device)

#         # 1) start from text embeddings
#         inputs_embeds = self.qwen.get_input_embeddings()(input_ids)

#         # 2) scatter RGB image embeddings into <|image_pad|>
#         if pixel_values is not None:
#             rgb_embeds_list, _ = self.qwen.get_image_features(pixel_values.to(device), image_grid_thw.to(device))
#             rgb_embeds = torch.cat(rgb_embeds_list, dim=0).to(inputs_embeds.device, inputs_embeds.dtype)  # (num_rgb_tokens, hidden)
#             rgb_mask = (input_ids == self.qwen.config.image_token_id).unsqueeze(-1).expand_as(inputs_embeds)
#             if inputs_embeds[rgb_mask].numel() != rgb_embeds.numel():
#                 raise ValueError(f"RGB tokens/features mismatch: tokens_numel={inputs_embeds[rgb_mask].numel()} vs feats_numel={rgb_embeds.numel()}")
#             inputs_embeds = inputs_embeds.masked_scatter(rgb_mask, rgb_embeds)

#         # 3) scatter HS embeddings into <|hs_image_pad|>
#         if pixel_values_hs is not None:
#             hs_embeds, _ = self.hs_visual(pixel_values_hs.to(device), grid_thw=hs_grid_thw.to(device))  # (num_hs_tokens, hidden)
#             hs_embeds = hs_embeds.to(inputs_embeds.device, inputs_embeds.dtype)
#             hs_mask = (input_ids == self.hs_token_id).unsqueeze(-1).expand_as(inputs_embeds)
#             if inputs_embeds[hs_mask].numel() != hs_embeds.numel():
#                 raise ValueError(f"HS tokens/features mismatch: tokens_numel={inputs_embeds[hs_mask].numel()} vs feats_numel={hs_embeds.numel()}")
#             inputs_embeds = inputs_embeds.masked_scatter(hs_mask, hs_embeds)

#         # 4) compute RoPE position_ids: treat HS tokens as image tokens for indexing
#         rope_input_ids = input_ids.clone()
#         rope_input_ids[rope_input_ids == self.hs_token_id] = self.qwen.config.image_token_id

#         # IMPORTANT: image_grid_thw must include BOTH rgb and hs "images" in the same order as they appear in the prompt.
#         combined_grid = None
#         if image_grid_thw is not None and hs_grid_thw is not None:
#             combined_grid = torch.cat([image_grid_thw.to(device), hs_grid_thw.to(device)], dim=0)
#         elif image_grid_thw is not None:
#             combined_grid = image_grid_thw.to(device)
#         elif hs_grid_thw is not None:
#             combined_grid = hs_grid_thw.to(device)

#         position_ids, _ = self.qwen.model.get_rope_index(
#             input_ids=rope_input_ids,
#             image_grid_thw=combined_grid,
#             video_grid_thw=None,
#             attention_mask=attention_mask,
#         )

#         # 5) run language model directly (we already injected visual features)
#         lm_out = self.qwen.language_model(
#             inputs_embeds=inputs_embeds,
#             attention_mask=attention_mask,
#             position_ids=position_ids,
#             use_cache=False,
#         )
#         hidden_states = lm_out.last_hidden_state
#         logits = self.qwen.lm_head(hidden_states)

#         loss = None
#         if labels is not None:
#             loss = self.qwen.loss_function(logits=logits, labels=labels, vocab_size=self.qwen.config.text_config.vocab_size)

#         return {"loss": loss, "logits": logits}

In [24]:
# def num_tokens_from_grid(grid_thw, merge_size):
#     # grid_thw: (num_images,3) where (t,h,w) are patch-grid sizes
#     return int((grid_thw[0].prod() // (merge_size**2)).item())

# def build_dual_prompt(processor, question, n_rgb_tokens, n_hs_tokens, hs_token=HS_TOKEN):
#     vs = processor.vision_start_token
#     ve = processor.vision_end_token
#     rgb_block = vs + (processor.image_token * n_rgb_tokens) + ve
#     hs_block  = vs + (hs_token * n_hs_tokens) + ve
#     return f"{rgb_block}\n{hs_block}\n{question}"

In [25]:
# # ---- Load ONE hyperspectral cube (raw) ----
# hs_raw = np.load(rad_files[0]).astype(np.float32)          # (256,256,285) raw
# hs_raw = torch.from_numpy(hs_raw).permute(2,0,1)           # (285,256,256)

# # ---- Make a pseudo-RGB from 3 selected bands ----
# # set these indices to the bands you want to treat as R,G,B (default is 36, 22, 10)
# R_BAND, G_BAND, B_BAND = 36, 22, 10

# rgb = torch.stack([hs_raw[R_BAND], hs_raw[G_BAND], hs_raw[B_BAND]], dim=0)  # (3,256,256)

# # handle invalids and scale to [0,1] for Qwen image_processor
# valid_rgb = torch.isfinite(rgb) & (rgb != FILL)
# rgb = torch.where(valid_rgb, rgb, torch.zeros_like(rgb))

# # robust per-image scaling (keeps contrast; avoids -9999 issues)
# vals = rgb[valid_rgb]
# lo, hi = torch.quantile(vals, torch.tensor([0.01, 0.99], device=vals.device)) if vals.numel() else (0.0, 1.0)
# rgb01 = (rgb - lo) / (hi - lo + 1e-6)
# rgb01 = rgb01.clamp(0, 1)                                  # (3,256,256), float

# # ---- Feed pseudo-RGB through Qwen's normal vision preprocessor ----
# rgb_inputs = processor.image_processor(images=rgb01, return_tensors="pt")   # accepts torch tensors
# pixel_values = rgb_inputs["pixel_values"]                                   # (1,3,H,W) after resize/normalize
# image_grid_thw = rgb_inputs["image_grid_thw"]                               # (1,3)

# merge_rgb = processor.image_processor.merge_size
# n_rgb = num_tokens_from_grid(image_grid_thw, merge_rgb)

# # ---- HS side (your custom encoder) ----
# hs = hs_raw                                                  # (285,256,256)
# valid_hs = (hs != FILL) & torch.isfinite(hs)
# hs = torch.where(
#     valid_hs,
#     (hs - torch.tensor(band_mean).view(-1,1,1)) / torch.tensor(band_std).view(-1,1,1),
#     torch.zeros_like(hs),
# )
# pixel_values_hs = hs.unsqueeze(0)                             # (1,285,256,256)

# hs_grid_thw = torch.tensor([[1, 256//16, 256//16]], dtype=torch.long)  # [[1,16,16]]
# n_hs = int((hs_grid_thw[0].prod() // (2**2)).item())          # if HS encoder spatial_merge_size=2

# # ---- Text / tokens ----
# question = "Compare the two inputs. What stands out?"
# prompt = build_dual_prompt(processor, question, n_rgb, n_hs)
# tok = processor.tokenizer(prompt, return_tensors="pt")

# # ---- Build dual model + run ----
# hs_encoder = model.enc  # HyperspectralAutoencoder(...).enc
# dual = DualVisionQwen(qwen, hs_encoder, HS_TOKEN_ID).cuda()

# out = dual(
#     input_ids=tok["input_ids"],
#     attention_mask=tok.get("attention_mask", None),
#     pixel_values=pixel_values,
#     image_grid_thw=image_grid_thw,
#     pixel_values_hs=pixel_values_hs,
#     hs_grid_thw=hs_grid_thw,
# )
# print(out["logits"].shape)